# ROCLING 2026 DSA — 實驗 9：L3 可控生成增強（高/低 arousal 合成 + 偽標）

對新住民自我反思文本預測 valence / arousal (1-9)。評分：V/A 各 MAE(↓)+PCC(↑)，4 指標 mean rank。

> **本 notebook = `current_pipeline.md` 待跑清單的實驗 9**：用 L3 情感知識圖譜 × LLM 生成「特定 arousal 的新住民文本」，補我們最痛的 **arousal 分布壓縮**。這是唯一能「補缺目標標籤」的招。

## 與 `Rocling2026_Colab.ipynb` 的差別
| | 主 notebook | 本 notebook (aug) |
|---|---|---|
| 定位 | L2→L3→L1 全流程 | 專跑**實驗 9 增強**（實驗 4 的加強版）|
| 增強資料 | 需現場呼叫 API 生成 | **已附 `train_aug.csv`（400 篇）直接用**，不花 API 費 |
| 流程 | install→上傳→L2→L3→生成→訓練 | install→上傳→合併資料→看分布→訓練 |

`baseline_aug.zip` 已內含生成好的 `data/train_aug.csv`（本機跨 5 個 VA 區間各 80 篇、去重過）。

**使用前**：上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**，然後依序執行。

In [ ]:
# 1) 安裝套件（訓練用；transformers。只有要「重生成」或「偽標」才需 anthropic/pydantic）
!pip -q install "transformers>=4.40" scikit-learn scipy
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) 上傳 baseline_aug.zip（已含 data/train_aug.csv）並解壓、切目錄
from google.colab import files
up = files.upload()              # 選 baseline_aug.zip
import zipfile, os
zname = list(up.keys())[0]
with zipfile.ZipFile(zname) as z: z.extractall('.')
for root, _, fs in os.walk('.'):
    if 'train.py' in fs and 'data' in os.listdir(root):
        os.chdir(root); break
os.makedirs('outputs', exist_ok=True)
print('工作目錄:', os.getcwd())
print('train_aug.csv 存在:', os.path.exists('data/train_aug.csv'))

## 合併增強資料進 train.csv

把 400 篇合成文本併進 `train.csv`。以 `train_base.csv`（原始備份）為基底重組，**可重跑不疊加**。

In [ ]:
# 3) 合併：train.csv = train_base.csv（原始） + train_aug.csv（增強）
#    SHRINK<1 把增強標籤向「真實訓練集 V/A 平均」收縮，修實驗 9 的 A_MAE 爆掉問題：
#    new = μ + SHRINK*(bin中心 − μ)。SHRINK=1.0→原始實驗9；建議先試 0.6（=實驗9b）。
import pandas as pd, os, shutil
SHRINK = 0.6
base = 'data/train_base.csv'
if not os.path.exists(base):
    shutil.copy('data/train.csv', base)   # 第一次先備份原始訓練集
    print('已備份原始訓練集 →', base)
df_base = pd.read_csv(base)
df_aug  = pd.read_csv('data/train_aug.csv').copy()
if SHRINK < 1.0:
    a0 = df_aug.arousal.std()
    for col in ['valence', 'arousal']:
        mu = df_base[col].mean()
        df_aug[col] = (mu + SHRINK*(df_aug[col] - mu)).clip(1, 9).round(2)
    print(f'標籤收縮 SHRINK={SHRINK}：aug A std {a0:.2f} → {df_aug.arousal.std():.2f}（真實={df_base.arousal.std():.2f}）')
df = pd.concat([df_base, df_aug], ignore_index=True)
df.to_csv('data/train.csv', index=False)
print(f'train.csv = {len(df_base)}（原始） + {len(df_aug)}（增強） = {len(df)} 筆  (SHRINK={SHRINK})')


In [ ]:
# 4) 看增強前後的 arousal 分布（這步的重點：把被壓縮的 A 分布拉開）
import pandas as pd
b = pd.read_csv('data/train_base.csv'); a = pd.read_csv('data/train_aug.csv'); m = pd.read_csv('data/train.csv')
for name, d in [('原始', b), ('增強 400', a), ('合併後', m)]:
    print(f'{name:6s} N={len(d):5d}  A mean={d.arousal.mean():.2f} std={d.arousal.std():.2f}  '
          f'A≥6={100*(d.arousal>=6).mean():.1f}%  A≤3.5={100*(d.arousal<=3.5).mean():.1f}%')
# 簡易直方圖
import numpy as np
for name, d in [('原始', b), ('合併後', m)]:
    h, _ = np.histogram(d.arousal, bins=np.arange(1, 10.5, 1))
    bars = ' '.join(f'{int(100*x/len(d)):2d}' for x in h)
    print(f'{name:6s} A各整數區%(1..9): {bars}')

## 訓練實驗 9：MacBERT + 詞典特徵融合（實驗 4 架構，資料換成增強版）

詞典融合預設開啟（`external/emobank` 的 CVAW/CVAP）。產出 `outputs/best_model.pt` + `outputs/submission.csv`。

> 與實驗 4 的**唯一差異**：train.csv 多了 400 篇高/低 arousal 合成文本。dev/超參全部不變，可與實驗 4 公平比較。

In [ ]:
# 5) 訓練實驗 9（超參同實驗 4）
!python train.py --epochs 4 --batch_size 32 --model hfl/chinese-macbert-base

In [ ]:
# 6) 檢視 submission
import pandas as pd
df = pd.read_csv('outputs/submission.csv')
print(df.describe()); df.head(10)

In [ ]:
# 7) 下載訓練好的模型 + submission 回本機
from google.colab import files
files.download('outputs/submission.csv')
files.download('outputs/best_model.pt')

---
## （可選）偽標精修：teacher 重標 + 離群移除

對應 `current_pipeline.md` 的「BERT teacher 偽標 + 離群移除 (mean±1.5SD)」（借鑑 CYUT 冠軍流程）。

現在 `train_aug.csv` 的標籤是**bin 中心值 ± jitter**（粗但一致）。這段用剛訓好的模型當 teacher 重新預測增強文本的 VA，與 bin 標籤加權混合，並以每個 bin 的 mean±1.5SD 刪掉離群，再重訓一次。**會多訓練一輪，可選跑。**

In [ ]:
# 8) （可選）離群移除 + teacher 偽標精修
# 「離群移除」可直接跑（只用現有 bin 標籤）；「teacher 偽標」是說明用的虛擬碼。
import pandas as pd, numpy as np

AUG = 'data/train_aug.csv'
aug = pd.read_csv(AUG)

# --- A. 每個 bin 以 mean±1.5SD 刪離群（CYUT 冠軍流程；可直接執行）---
def binid(v, a):
    return (round(v), a >= 5)            # 粗分：V 整數 × A 高低
keep = np.ones(len(aug), bool)
for key in set(binid(v, a) for v, a in zip(aug.valence, aug.arousal)):
    idx = [i for i, (v, a) in enumerate(zip(aug.valence, aug.arousal)) if binid(v, a) == key]
    for col in ['valence', 'arousal']:
        x = aug[col].values[idx]; mu, sd = x.mean(), x.std() + 1e-6
        for j, i in enumerate(idx):
            if abs(x[j] - mu) > 1.5 * sd: keep[i] = False
print(f'離群移除：保留 {keep.sum()}/{len(aug)} 篇')
aug[keep].to_csv('data/train_aug_refined.csv', index=False)

# --- B. teacher 偽標（虛擬碼；需自寫一個吃 VARegressor + 詞典特徵的推論函式）---
#   train.py 的模型是 VARegressor(model_name, lex_dim)，推論要重建 tokenizer + lexicon 特徵，
#   非單行可得。若要做：載入 outputs/best_model.pt → 預測 aug['text'] 的 (pv, pa) →
#   aug['valence'] = 0.5*aug['valence'] + 0.5*pv（bin 標籤與偽標加權）→ 再跑上面的離群移除。
#
# 要用精修版重訓：把 cell 3 的 train_aug.csv 換成 train_aug_refined.csv，再跑一次 cell 3、5。


---
## （可選）重新生成增強資料（需 API，會計費）

已附的 `train_aug.csv` 是本機生成好的，一般**不需重生**。若要調參或加量再生一批：需先有 `outputs/l3_graph.pkl`（請先跑主 notebook 的 L2→L3），並設定 API key。

In [ ]:
# 9) （可選）重生成：需 outputs/l3_graph.pkl + API key
# !pip -q install anthropic openai pydantic gensim jieba networkx
# import os
# from google.colab import userdata
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')   # 或 OPENAI_API_KEY
# # Claude（預設）：每區間 80 篇、已含近似去重；不加 --append，先生到 train_aug.csv 再人工抽檢
# !python augment_generate.py --per_bin 80
# # OpenAI 改用：!python augment_generate.py --provider openai --per_bin 80
# from google.colab import files; files.download('data/train_aug.csv')